# Task 1: Coding

## Task 1-A: Basic Decision Tree Learning Procedure

**Target Transformation Explanation:**
The original dataset contains a `quality` column with integer scores ranging from 3 to 8. To transform this into a binary classification task as required, I defined a threshold: wines with a quality score of 7 or higher are labeled as `1` (Good), and those below 7 are labeled as `0` (Normal/Bad). This approach simplifies the learning process for our decision tree, preventing it from creating too complex or fragmented rules to predict 6 different specific integer values, which would likely lead to overfitting and tiny leaf nodes.

**Train/Test Split Choice:**
I have chosen a standard 80/20 random split for the training and test sets. 80% of the data provides a sufficiently large and diverse sample size (approx. 1280 examples) for the decision tree to learn the `if-then-else` splitting rules effectively. The remaining 20% (approx. 320 examples) is completely hidden during training and serves as an unbiased evaluation set to test the model's generalization capabilities on unseen data.

In [ ]:
import pandas as pd
import numpy as np

# 1. Ensure reproducibility as strictly required by the assignment

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def load_and_transform_data(filepath):
    """
    Loads the wine quality dataset and transforms the target into a binary class.
    """
    # Note: The data uses semicolons as separators
    df = pd.read_csv(filepath, sep=';')

    # Transform target: 1 if quality >= 7, else 0
    df['target'] = (df['quality'] >= 7).astype(int)

    # Drop the original 'quality' column as it's replaced by 'target'
    df = df.drop('quality', axis=1)

    return df

def train_test_split_custom(df, test_size=0.2):
    """
    Custom implementation of train-test split to avoid using external ML packages.
    """
    # Shuffle the indices randomly
    shuffled_indices = np.random.permutation(len(df))

    # Calculate the number of test samples
    test_set_size = int(len(df) * test_size)

    # Split the indices
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]

    # Create the train and test DataFrames
    train_df = df.iloc[train_indices].reset_index(drop=True)
    test_df = df.iloc[test_indices].reset_index(drop=True)

    return train_df, test_df

# --- Execution ---
# Making sure 'winequality-red.csv' is uploaded to environment
try:
    wine_data = load_and_transform_data('winequality-red.csv')
    train_data, test_data = train_test_split_custom(wine_data, test_size=0.2)

    print(f"Total dataset shape: {wine_data.shape}")
    print(f"Training set shape: {train_data.shape}")
    print(f"Test set shape: {test_data.shape}")
    print("\nTarget distribution in Training Set:")
    print(train_data['target'].value_counts())
except FileNotFoundError:
    print("Please upload 'winequality-red.csv' to the files section.")

Total dataset shape: (1599, 12)
Training set shape: (1280, 12)
Test set shape: (319, 12)

Target distribution in Training Set:
target
0    1110
1     170
Name: count, dtype: int64


## Task 1-A: Splitting Criterion - Information Gain

**Mathematical Foundation:**
To decide on the best splits, this decision tree implementation uses the **Information Gain** criterion as discussed in the lectures.
1. **Entropy**: Measures the impurity or randomness of a dataset. The formula is $entropy(s) = -p_{pos} \log_2 p_{pos} - p_{neg} \log_2 p_{neg}$. A pure dataset (only one class) has an entropy of 0.
2. **Information Gain**: Calculates the reduction in entropy after a dataset is split on an attribute. The formula used is $IG = entropy(parent) - \frac{n_{left}}{n} entropy(left\_child) - \frac{n_{right}}{n} entropy(right\_child)$. The algorithm will iterate through all possible features and thresholds to find the split that maximizes this information gain.

In [ ]:
def calculate_entropy(y):
    """
    Calculates the entropy of a label array y.
    Formula: - sum(p * log2(p))
    """
    # Count the occurrences of each class (0 and 1)
    counts = np.bincount(y)
    # Calculate probabilities
    probabilities = counts / len(y)

    entropy = 0
    for p in probabilities:
        if p > 0: # Avoid log2(0) which is undefined
            entropy -= p * np.log2(p)

    return entropy

def calculate_information_gain(y, y_left, y_right):
    """
    Calculates the Information Gain of a split.
    Formula: Entropy(parent) - [Weighted Average * Entropy(children)]
    """
    # 1. Calculate the entropy of the parent node
    parent_entropy = calculate_entropy(y)

    # 2. Calculate the weights for the child nodes
    n = len(y)
    n_left = len(y_left)
    n_right = len(y_right)

    # If a split results in an empty child, it's a useless split, info gain is 0
    if n_left == 0 or n_right == 0:
        return 0

    # 3. Calculate the weighted average entropy of the children
    child_entropy = (n_left / n) * calculate_entropy(y_left) + (n_right / n) * calculate_entropy(y_right)

    # 4. Calculate Information Gain
    ig = parent_entropy - child_entropy
    return ig

# --- Quick Test ---
# Let's test it with a dummy split to ensure it works
dummy_y = np.array([1, 1, 1, 0, 0, 0]) # Mixed labels
dummy_y_left = np.array([1, 1, 1])     # Perfectly pure
dummy_y_right = np.array([0, 0, 0])    # Perfectly pure

print(f"Parent Entropy: {calculate_entropy(dummy_y):.4f}")
print(f"Information Gain from perfect split: {calculate_information_gain(dummy_y, dummy_y_left, dummy_y_right):.4f}")

Parent Entropy: 1.0000
Information Gain from perfect split: 1.0000


## Task 1-A: Finding the Best Split

**Search and Score Methodology:**
Following the "Greedy Recursive Splitting" algorithm taught in the lectures, the model must find the best feature and threshold to split the data at each node. The `get_best_split` function iterates over all available features (e.g., alcohol, pH, sulphates). For each feature, it extracts all unique values present in the training data to use as potential thresholds, as testing other values is unnecessary. It then temporarily splits the dataset based on each threshold and calculates the Information Gain. The split that yields the maximum Information Gain is selected as the optimal splitting rule for that node.

In [ ]:
def split_dataset(X, y, feature_index, threshold):
    """
    Splits the dataset into left and right branches based on a feature and threshold.
    Left: feature value <= threshold
    Right: feature value > threshold
    """
    # Create boolean masks for filtering
    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold

    return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

def get_best_split(X, y):
    """
    Iterates over all features and unique thresholds to find the split
    that maximizes Information Gain.
    """
    best_split = {}
    max_info_gain = -1
    n_features = X.shape[1]

    # Iterate over all features (columns)
    for feature_index in range(n_features):
        # Get all unique values for this feature to use as candidate thresholds
        #
        thresholds = np.unique(X[:, feature_index])

        # Iterate over all candidate thresholds
        for threshold in thresholds:
            # Split the data
            X_left, y_left, X_right, y_right = split_dataset(X, y, feature_index, threshold)

            # Skip if the split doesn't actually divide the data (e.g., all data goes to one side)
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            # Calculate Information Gain
            current_info_gain = calculate_information_gain(y, y_left, y_right)

            # Update best split if current is better
            if current_info_gain > max_info_gain:
                max_info_gain = current_info_gain
                best_split = {
                    'feature_index': feature_index,
                    'threshold': threshold,
                    'X_left': X_left,
                    'y_left': y_left,
                    'X_right': X_right,
                    'y_right': y_right,
                    'info_gain': current_info_gain
                }

    return best_split

# --- Quick Test on Real Data ---
# Convert pandas dataframe to numpy arrays for faster computation
X_train_np = train_data.drop('target', axis=1).values
y_train_np = train_data['target'].values
feature_names = train_data.drop('target', axis=1).columns.tolist()

print("Searching for the first best split (Root Node)...")
best_root_split = get_best_split(X_train_np, y_train_np)

best_feature_name = feature_names[best_root_split['feature_index']]
print(f"Best Feature to split on: {best_feature_name}")
print(f"Best Threshold: {best_root_split['threshold']}")
print(f"Information Gain achieved: {best_root_split['info_gain']:.4f}")
print(f"Left branch size: {len(best_root_split['y_left'])}, Right branch size: {len(best_root_split['y_right'])}")

Searching for the first best split (Root Node)...
Best Feature to split on: alcohol
Best Threshold: 11.5
Information Gain achieved: 0.0869
Left branch size: 1086, Right branch size: 194


## Task 1-A & 1-B: Recursive Tree Building and Depth Control

**Algorithm Implementation:**
The `DecisionTree` class implements the "Greedy Recursive Splitting" algorithm.
* **Node Structure:** Each split is stored as a `Node` object, containing the feature index, threshold, and pointers to its left and right child nodes. If it's a leaf node, it stores the majority class prediction.
* **Base Cases (Stopping Criteria):** The recursive `build_tree` function stops and creates a leaf node when:
    1. All labels in the current branch belong to the same class (pure node).
    2. The `stopping_depth` limit is reached (implementing Task 1-B for complexity control).
    3. No further Information Gain can be achieved.
* **Leaf Value:** When a leaf is formed, it predicts the majority class (`mode`) of the samples that reached that node.

In [ ]:
class Node:
    """
    A class representing a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, info_gain=None, value=None):
        # For decision nodes (internal nodes)
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.info_gain = info_gain

        # For leaf nodes
        self.value = value

class DecisionTree:
    """
    Custom Decision Tree Classifier.
    """
    def __init__(self, stopping_depth=None):
        # Task 1-B: Implement stopping_depth
        self.root = None
        self.stopping_depth = stopping_depth

    def build_tree(self, X, y, current_depth=0):
        """
        Recursive function to build the tree.
        """
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        # --- Stopping Criteria ---
        # 1. If only one class remains (pure node)
        # 2. If max depth is reached
        if len(unique_classes) == 1 or (self.stopping_depth is not None and current_depth >= self.stopping_depth):
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        # --- Find Best Split ---
        best_split = get_best_split(X, y)

        # If we found a valid split with Information Gain > 0
        if best_split and best_split['info_gain'] > 0:
            # Recursive calls for left and right branches
            left_subtree = self.build_tree(best_split['X_left'], best_split['y_left'], current_depth + 1)
            right_subtree = self.build_tree(best_split['X_right'], best_split['y_right'], current_depth + 1)

            # Return decision node
            return Node(best_split['feature_index'], best_split['threshold'],
                        left_subtree, right_subtree, best_split['info_gain'])

        # If no split improves information gain, make it a leaf
        leaf_value = self._calculate_leaf_value(y)
        return Node(value=leaf_value)

    def _calculate_leaf_value(self, y):
        """
        Returns the most common class in y (Majority vote / Mode).

        """
        # np.bincount counts occurrences, argmax finds the index (class) with max count
        return np.bincount(y).argmax()

    def train(self, X, y):
        """
        Starts the recursive tree building process.
        """
        self.root = self.build_tree(X, y)

    def print_tree(self, node=None, indent=" ", feature_names=None):
        """
        Helper function to print the if-then-else rules of the tree.

        """
        if not node:
            node = self.root

        # If it's a leaf node
        if node.value is not None:
            # 1 is Good Wine, 0 is Normal/Bad Wine
            class_name = "Good Wine (1)" if node.value == 1 else "Normal Wine (0)"
            print(f"{class_name}")
        # If it's a decision node
        else:
            feature_name = feature_names[node.feature_index] if feature_names else f"Feature {node.feature_index}"
            print(f"If {feature_name} <= {node.threshold} ? (Info Gain: {node.info_gain:.4f})")
            print(f"{indent}├─ Yes: ", end="")
            self.print_tree(node.left, indent + "│  ", feature_names)
            print(f"{indent}└─ No:  ", end="")
            self.print_tree(node.right, indent + "   ", feature_names)

# --- Quick Test ---
print("Training the Decision Tree with a max depth of 2 (to keep the printout small)...")
# Initialize tree with stopping_depth=2 for Task 1-B test
tree = DecisionTree(stopping_depth=2)
tree.train(X_train_np, y_train_np)

print("\n--- Decision Tree Rules ---")
tree.print_tree(feature_names=feature_names)

Training the Decision Tree with a max depth of 2 (to keep the printout small)...

--- Decision Tree Rules ---
If alcohol <= 11.5 ? (Info Gain: 0.0869)
 ├─ Yes: If sulphates <= 0.64 ? (Info Gain: 0.0432)
 │  ├─ Yes: Normal Wine (0)
 │  └─ No:  Normal Wine (0)
 └─ No:  If sulphates <= 0.68 ? (Info Gain: 0.1302)
    ├─ Yes: Normal Wine (0)
    └─ No:  Good Wine (1)


## Task 1-B: Discussion on Depth Control (Level 2, 3, 4)
When analyzing the printed tree at `stopping_depth=2`, I observed that some branches (like `alcohol <= 11.5`) result in leaf nodes that predict the same class (Normal Wine) for both outcomes, despite having a positive Information Gain. This happens because Information Gain measures impurity reduction, not a change in the majority class. At depth 2, the tree hasn't grown deep enough to isolate the minority "Good Wine" class in that specific sub-population. When increasing the depth to 3 and 4, the tree creates more specific, granular rules, allowing those concentrated pockets of "Good Wine" to finally form their own pure leaf nodes, demonstrating how depth controls the model's complexity and ability to capture finer patterns.

## Task 1-C: Test Procedure and Evaluation
The test procedure recursively traverses the built decision tree for each new unseen data point. At each decision node, it evaluates the feature threshold and routes the instance left or right until it reaches a leaf node, returning that leaf's class prediction. To evaluate the classifier, I implemented the **Accuracy** metric, which calculates the percentage of correctly predicted labels on the unseen Test Set.

In [ ]:
# --- Task 1-C: Test Procedure ---

def predict_single_instance(node, x):
    """
    Recursively traverses the tree to predict the class for a single instance x.
    """
    # If it's a leaf node, return its value
    if node.value is not None:
        return node.value

    # If it's a decision node, traverse left or right
    feature_value = x[node.feature_index]
    if feature_value <= node.threshold:
        return predict_single_instance(node.left, x)
    else:
        return predict_single_instance(node.right, x)

def predict(tree, X):
    """
    Predicts labels for a dataset X using the trained tree.
    """
    predictions = [predict_single_instance(tree.root, x) for x in X]
    return np.array(predictions)

def calculate_accuracy(y_true, y_pred):
    """
    Evaluation Metric: Accuracy
    """
    correct_predictions = np.sum(y_true == y_pred)
    total_predictions = len(y_true)
    return correct_predictions / total_predictions

# --- Execution & Evaluation for Depths 2, 3, and 4 ---
X_test_np = test_data.drop('target', axis=1).values
y_test_np = test_data['target'].values

depths_to_test = [2, 3, 4]

for depth in depths_to_test:
    print(f"\n=========================================")
    print(f"Evaluating Decision Tree at Depth: {depth}")
    print(f"=========================================")

    # Train
    current_tree = DecisionTree(stopping_depth=depth)
    current_tree.train(X_train_np, y_train_np)

    # Predict on Training Data (to see how well it learned)
    train_predictions = predict(current_tree, X_train_np)
    train_accuracy = calculate_accuracy(y_train_np, train_predictions)

    # Predict on Unseen Test Data (Task 1-C Evaluation)
    test_predictions = predict(current_tree, X_test_np)
    test_accuracy = calculate_accuracy(y_test_np, test_predictions)

    print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
    print(f"Test Accuracy:     {test_accuracy * 100:.2f}%")

    # Print the tree structure for discussion purposes
    print("\nTree Structure:")
    current_tree.print_tree(feature_names=feature_names)


Evaluating Decision Tree at Depth: 2
Training Accuracy: 89.22%
Test Accuracy:     86.52%

Tree Structure:
If alcohol <= 11.5 ? (Info Gain: 0.0869)
 ├─ Yes: If sulphates <= 0.64 ? (Info Gain: 0.0432)
 │  ├─ Yes: Normal Wine (0)
 │  └─ No:  Normal Wine (0)
 └─ No:  If sulphates <= 0.68 ? (Info Gain: 0.1302)
    ├─ Yes: Normal Wine (0)
    └─ No:  Good Wine (1)

Evaluating Decision Tree at Depth: 3
Training Accuracy: 90.16%
Test Accuracy:     85.89%

Tree Structure:
If alcohol <= 11.5 ? (Info Gain: 0.0869)
 ├─ Yes: If sulphates <= 0.64 ? (Info Gain: 0.0432)
 │  ├─ Yes: If total sulfur dioxide <= 45.0 ? (Info Gain: 0.0186)
 │  │  ├─ Yes: Normal Wine (0)
 │  │  └─ No:  Normal Wine (0)
 │  └─ No:  If alcohol <= 9.8 ? (Info Gain: 0.0583)
 │     ├─ Yes: Normal Wine (0)
 │     └─ No:  Normal Wine (0)
 └─ No:  If sulphates <= 0.68 ? (Info Gain: 0.1302)
    ├─ Yes: If total sulfur dioxide <= 15.0 ? (Info Gain: 0.1349)
    │  ├─ Yes: Good Wine (1)
    │  └─ No:  Normal Wine (0)
    └─ No:  If fre

# Task 2: Reflection

### A. Changing the Splitting Criterion
If we decide to change the splitting criterion from **Information Gain** to another metric, the most common alternative is the **Gini Impurity** (or Gini Index).
* **Explanation:** While Information Gain uses entropy (logarithmic calculations) to measure the reduction in randomness, Gini Impurity measures the probability that a randomly chosen element would be incorrectly classified if it were randomly labeled according to the distribution of labels in the node.
* **Impact on the Decision Tree:** Mathematically, Information Gain tends to favor splits that result in a larger number of smaller, highly pure nodes, whereas Gini tends to isolate the most frequent class in one large branch. Changing to Gini might alter the specific features chosen for the splits (e.g., the root node might change if Gini evaluates another feature's impurity reduction differently), resulting in a different tree structure. However, in practical applications, both criteria often yield trees with very similar predictive performance. Computationally, Gini is slightly faster because it avoids logarithmic functions.

### B. Detecting Overfitting and Underfitting using the Test Procedure
Yes, the test procedure implemented in Task 1-C is a robust and direct method for indicating whether the decision tree is over- or underfitting.
* **Explanation:** By comparing the `Training Accuracy` and the `Test Accuracy`, we can diagnose the model's state:
    * **Underfitting:** If both the Training Accuracy and Test Accuracy are low (e.g., if we limited the tree to `stopping_depth=1`), it indicates the model is too simple to capture the underlying patterns in the data.
    * **Overfitting:** If the Training Accuracy is extremely high (e.g., approaching 99% at a depth of 15) but the Test Accuracy stagnates or drops significantly, it means the tree has memorized the training data's noise rather than learning generalizable rules.
* **Evidence from our results:** We can observe a hint of this in our outputs. From depth 2 to 3, the training accuracy increased (89.22% to 90.16%), but the test accuracy slightly dropped (86.52% to 85.89%). This divergence between training and testing performance is the classic signature of the onset of overfitting, proving that our 1-C procedure effectively monitors model generalization.

In [ ]:
!jupyter nbconvert --to html CS361_HW1_PAN.ipynb